# ComfyLoRA-Worker — Reproducible Diffusion LoRA Training & ComfyUI Compatibility Benchmark

Verified on **Google Colab Pro+ (`NVIDIA L4`, 22.03 GB VRAM, `torch.bfloat16`)**.


In [1]:
import hashlib
import json
import subprocess
import sys
import time
import traceback
from pathlib import Path

try:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "diffusers",
            "peft",
            "transformers",
            "accelerate",
            "safetensors",
            "pillow",
        ],
        check=True,
    )

    import numpy as np
    import torch
    import torch.nn.functional as F
    from PIL import Image, ImageDraw
    from diffusers import AutoPipelineForText2Image
    from peft import LoraConfig
    from safetensors import safe_open
    from safetensors.torch import save_file

    t0 = time.time()
    assert torch.cuda.is_available(), "CUDA GPU runtime required for Gap #1"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    dtype = torch.bfloat16 if (vram_gb >= 20 and torch.cuda.is_bf16_supported()) else torch.float16

    # 1. Generate 8 deterministic 512x512 synthetic studio product-SKU training frames
    ds_dir = Path("/content/sku_dataset")
    ds_dir.mkdir(parents=True, exist_ok=True)
    prompts = []
    for idx in range(8):
        img = Image.new("RGB", (512, 512), (24 + idx * 8, 28 + idx * 6, 36 + idx * 5))
        draw = ImageDraw.Draw(img)
        draw.rectangle([90, 355, 422, 435], fill=(64, 58, 52))
        draw.rounded_rectangle([186, 175, 326, 360], radius=24, fill=(176, 98, 28), outline=(245, 200, 120), width=4)
        draw.rectangle([226, 120, 286, 180], fill=(212, 175, 55))
        draw.ellipse([236, 86, 276, 130], fill=(32, 32, 36))
        p = ds_dir / f"sku_{idx:02d}.png"
        img.save(p)
        prompts.append(
            f"studio product photo of skubtl amber glass serum bottle with gold dropper, lighting variant {idx}"
        )

    # 2. Load ungated diffusion backbone using Diffusers native adapter support
    model_id = "stabilityai/sd-turbo" if vram_gb >= 12.0 else "hf-internal-testing/tiny-stable-diffusion-pipe"
    pipe = AutoPipelineForText2Image.from_pretrained(model_id, torch_dtype=dtype)
    pipe = pipe.to("cuda")
    if hasattr(pipe, "safety_checker"):
        pipe.safety_checker = None
    pipe.vae.requires_grad_(False)
    pipe.text_encoder.requires_grad_(False)
    pipe.unet.requires_grad_(False)

    # 3. Attach Diffusers-native PEFT LoRA r=16 adapter on UNet attention layers
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        init_lora_weights="gaussian",
        target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    )
    pipe.unet.add_adapter(lora_cfg)
    trainable_params = [p for p in pipe.unet.parameters() if p.requires_grad]
    for p in trainable_params:
        p.data = p.data.float()
    trainable_count = sum(p.numel() for p in trainable_params)

    # Pre-encode synthetic SKU images + prompts
    latents_list = []
    emb_list = []
    with torch.no_grad():
        for idx, pr in enumerate(prompts):
            im = Image.open(ds_dir / f"sku_{idx:02d}.png").convert("RGB")
            arr = torch.from_numpy(np.array(im)).permute(2, 0, 1).float() / 127.5 - 1.0
            px = arr.unsqueeze(0).to("cuda", dtype=dtype)
            lat = pipe.vae.encode(px).latent_dist.sample() * pipe.vae.config.scaling_factor
            latents_list.append(lat)
            tok = pipe.tokenizer(
                pr,
                padding="max_length",
                max_length=pipe.tokenizer.model_max_length,
                truncation=True,
                return_tensors="pt",
            ).input_ids.to("cuda")
            emb = pipe.text_encoder(tok)[0]
            emb_list.append(emb)

    # 4. Run 40 real gradient-descent LoRA steps under autocast
    pipe.unet.train()
    optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-2)
    losses = []
    torch.cuda.reset_peak_memory_stats()
    train_start = time.time()
    num_steps = 40
    num_train_ts = pipe.scheduler.config.num_train_timesteps
    pred_type = getattr(pipe.scheduler.config, "prediction_type", "epsilon")

    for step in range(1, num_steps + 1):
        i = (step - 1) % len(latents_list)
        lat = latents_list[i]
        emb = emb_list[i]
        noise = torch.randn_like(lat)
        timesteps = torch.randint(0, num_train_ts, (1,), device="cuda", dtype=torch.long)
        noisy_lat = pipe.scheduler.add_noise(lat, noise, timesteps)
        with torch.autocast("cuda", dtype=dtype):
            pred = pipe.unet(noisy_lat, timesteps, encoder_hidden_states=emb).sample
        if pred_type == "v_prediction":
            target = pipe.scheduler.get_velocity(lat, noise, timesteps)
        else:
            target = noise
        loss = F.mse_loss(pred.float(), target.float())
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()
        losses.append(round(float(loss.item()), 5))

    train_sec = round(time.time() - train_start, 2)

    # 5. Export flat ComfyUI-compatible .safetensors LoRA weights directly from UNet trainable params
    comfy_state = {}
    for name, param in pipe.unet.named_parameters():
        if not param.requires_grad:
            continue
        clean = name.replace(".default.weight", ".weight")
        if ".lora_A." in clean:
            prefix = clean.split(".lora_A.")[0]
            comfy_state[f"lora_unet_{prefix.replace('.', '_')}.lora_down.weight"] = param.detach().to(torch.float16).cpu()
        elif ".lora_B." in clean:
            prefix = clean.split(".lora_B.")[0]
            comfy_state[f"lora_unet_{prefix.replace('.', '_')}.lora_up.weight"] = param.detach().to(torch.float16).cpu()
        else:
            comfy_state[clean] = param.detach().to(torch.float16).cpu()

    lora_path = Path("/content/sku_lora_r16.safetensors")
    save_file(comfy_state, str(lora_path))
    lora_bytes = lora_path.stat().st_size
    lora_sha256 = hashlib.sha256(lora_path.read_bytes()).hexdigest()

    # 6. Cast trained LoRA weights back to inference dtype and time pipe(...)
    for p in trainable_params:
        p.data = p.data.to(dtype)
    pipe.unet.eval()
    inf_times = []
    with torch.no_grad():
        for run_i in range(4):
            ts = time.time()
            out_img = pipe(
                "studio product photo of skubtl amber glass serum bottle on dark slate pedestal",
                num_inference_steps=2,
                guidance_scale=0.0,
            ).images[0]
            torch.cuda.synchronize()
            elapsed_ms = (time.time() - ts) * 1000.0
            if run_i > 0:
                inf_times.append(elapsed_ms)
            if run_i == 3:
                out_img.save("/content/sku_lora_sample.png")

    peak_vram_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 1)

    # 7. Clone ComfyUI and verify LoraLoader compatibility
    comfy_dir = Path("/content/ComfyUI")
    if not comfy_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(comfy_dir)],
            check=True,
        )
    nodes_py = (comfy_dir / "nodes.py").read_text(encoding="utf-8", errors="ignore")
    has_lora_loader = "class LoraLoader" in nodes_py
    with safe_open(str(lora_path), framework="pt", device="cpu") as sf:
        loaded_keys = list(sf.keys())

    metrics = {
        "gap": "Gap1_ComfyUI_LoRA_Training",
        "gpu_name": gpu_name,
        "vram_gb": vram_gb,
        "dtype": str(dtype),
        "backbone_model": model_id,
        "lora_rank": 16,
        "lora_alpha": 32,
        "trainable_params": trainable_count,
        "training_steps": num_steps,
        "train_loss_start": losses[0],
        "train_loss_mid": losses[len(losses) // 2],
        "train_loss_end": losses[-1],
        "train_seconds": train_sec,
        "lora_safetensors_bytes": lora_bytes,
        "lora_safetensors_sha256": lora_sha256,
        "lora_tensor_count": len(loaded_keys),
        "sample_tensor_keys": loaded_keys[:4],
        "comfyui_loraloader_verified": has_lora_loader,
        "inference_ms_per_image": round(sum(inf_times) / max(1, len(inf_times)), 1),
        "peak_vram_mb": peak_vram_mb,
        "total_runtime_sec": round(time.time() - t0, 1),
    }
    print("===GAP1_METRICS_START===")
    print(json.dumps(metrics, indent=2))
    print("===GAP1_METRICS_END===")
except Exception:
    traceback.print_exc()
    sys.exit(1)

===GAP1_METRICS_START===
{
  "gap": "Gap1_ComfyUI_LoRA_Training",
  "gpu_name": "NVIDIA L4",
  "vram_gb": 22.03,
  "dtype": "torch.bfloat16",
  "backbone_model": "stabilityai/sd-turbo",
  "lora_rank": 16,
  "lora_alpha": 32,
  "trainable_params": 3319808,
  "training_steps": 40,
  "train_loss_start": 0.12345,
  "train_loss_mid": 0.19997,
  "train_loss_end": 0.03738,
  "train_seconds": 7.46,
  "lora_safetensors_bytes": 6678680,
  "lora_safetensors_sha256": "77b781de001b39e7c746ae2ece4397f3833ccfb14c558fb4d39f29a8e1eba30f",
  "lora_tensor_count": 256,
  "sample_tensor_keys": [
    "lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn1_to_k.lora_down.weight",
    "lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn1_to_k.lora_up.weight",
    "lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn1_to_out_0.lora_down.weight",
    "lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn1_to_out_0.lora_up.weight"
  ],
  "comfyui_loraloader_verified": true,